In [ ]:
# Input here is the unseen material 
#

In [ ]:
# Chunk 0 - setup and imports
import gc
import json
from pathlib import Path

import cv2
import torch
from tqdm import tqdm
from ultralytics import YOLO


def find_repo_root(start=None, marker=".git"):
    path = Path(start or Path.cwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find a repo root (looking for '{marker}')")


REPO_ROOT = find_repo_root()

CONFIG = {
    "VIDEOS_DIR": REPO_ROOT / "use-a-crab-detector" / "data" / "videos",  # adjust to your BRUV footage
    # change model path if you are using any other
    "MODEL_PATH": REPO_ROOT / "use-a-crab-detector" / "model" / "pretrained-crab-detector" / "best.pt",
    "OUTPUT_DIR": REPO_ROOT / "use-a-crab-detector" / "data" / "predictions",
    "CONFIDENCE": 0.1,
    "IMAGE_SIZE": 960,
     # Set True to also save an annotated video per input, with boxes drawn
    # in, useful for spot-checking a few results by eye. Off by default
    # since it adds significant disk use and time across a full batch.
    "SAVE_VIDEOS": False,
}
CONFIG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".flv", ".wmv", ".m4v"}

In [ ]:
# Chunk 1 - finding the videos, and the amount
def find_video_files(videos_dir: Path) -> list:
    """Recursively find every video file under videos_dir."""
    return [p for p in Path(videos_dir).rglob("*") if p.suffix.lower() in VIDEO_EXTENSIONS]


video_files = find_video_files(CONFIG["VIDEOS_DIR"])
print(f"Found {len(video_files)} videos to process")

In [ ]:
# Chunk 2 - run the detector on every video
def run_detector_on_videos(video_files: list, config: dict) -> list:
    """
    Run the trained model on every video, sampling roughly one frame per
    second, and collect every detection into a single list matching the
    format used elsewhere in the repo:
    {"image_id": "<video_stem>_frame_<n>", "score": confidence, "bbox": [x, y, w, h]}.

    Videos that fail to open or error out mid-processing are skipped and
    reported at the end, rather than stopping the whole batch.
    """
    model = YOLO(str(config["MODEL_PATH"]))
    predictions = []
    failed_videos = []

    for video_path in tqdm(video_files, desc="Processing videos"):
        try:
            cap = cv2.VideoCapture(str(video_path))
            if not cap.isOpened():
                failed_videos.append((video_path.name, "Cannot open"))
                cap.release()
                continue

            fps = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
            # Fall back to a fixed stride if a video reports a broken FPS value
            vid_stride = max(1, round(fps)) if 0 < fps <= 120 else 30

            results = model.predict(
                source=str(video_path),
                conf=config["CONFIDENCE"],
                vid_stride=vid_stride,
                imgsz=config["IMAGE_SIZE"],
                stream=True,
                verbose=False,
                save=config["SAVE_VIDEOS"],
                project=str(config["OUTPUT_DIR"] / "annotated_videos") if config["SAVE_VIDEOS"] else None,
                name="run",
                exist_ok=True,
            )

            for frame_idx, result in enumerate(results):
                image_id = f"{video_path.stem}_frame_{frame_idx}"
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    predictions.append({
                        "image_id": image_id,
                        "score": float(box.conf[0]),
                        "bbox": [x1, y1, x2 - x1, y2 - y1],
                    })

            # Free GPU memory before moving to the next video
            del results
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            failed_videos.append((video_path.name, str(e)))
            continue

    print(f"\nProcessed {len(video_files) - len(failed_videos)}/{len(video_files)} videos")
    if failed_videos:
        print(f"{len(failed_videos)} videos failed:")
        for name, err in failed_videos[:10]:
            print(f"  {name}: {err}")

    return predictions

In [ ]:
# Chunk 3 - run and save
predictions = run_detector_on_videos(video_files, CONFIG)

output_path = CONFIG["OUTPUT_DIR"] / "predictions.json"
output_path.write_text(json.dumps(predictions, indent=2))
print(f"\nSaved {len(predictions)} detections to {output_path}")